In [3]:
!git clone https://github.com/sachadee/inswapper.git

fatal: destination path 'inswapper' already exists and is not an empty directory.


In [ ]:
!cd '/content/inswapper' && pip install -r requirements.txt

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copy the model from Google Drive to the checkpoint directory
!cd ..
!mkdir -p facesToSwap
!unzip /content/out_vidn.zip -d /content/facesToSwap
!mkdir -p SwappedFaces
!mkdir -p checkpoints
!cp '/content/drive/MyDrive/inswapper_128.onnx' './checkpoints/inswapper_128.onnx'

In [1]:
!git lfs install
!git clone https://huggingface.co/spaces/sczhou/CodeFormer

Git LFS initialized.
fatal: destination path 'CodeFormer' already exists and is not an empty directory.


In [9]:
import sys
from PIL import Image
sys.path.insert(0, '/content/inswapper')
from swapper import *
model = "/content/checkpoints/inswapper_128.onnx" # Changed to absolute path


source_img = [Image.open("/content/ana.jpg")]
target_img = Image.open("/content/facesToSwap/frame_10.jpg")


result_image = process(source_img, target_img, -1, -1, model)
result_image.save("result.png")

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:539 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': 

In [ ]:
import os
faces_to_swap_dir = "/content/facesToSwap"
swapped_faces_dir = "/content/SwappedFaces"

os.makedirs(swapped_faces_dir, exist_ok=True)

if os.path.exists("/content/ana.jpg"):
    print(f"Starting face swapping from '{faces_to_swap_dir}' to '{swapped_faces_dir}' using source face from 'ana.jpg")

    # Loop through images in the facesToSwap directory
    for filename in os.listdir(faces_to_swap_dir):
        # Check if the file is an image
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            target_image_path = os.path.join(faces_to_swap_dir, filename)
            output_image_path = os.path.join(swapped_faces_dir, filename)
            print(target_image_path,output_image_path)

            !python /content/inswapper/swapper.py \
                  --source_img="/content/ana.jpg" \
                  --target_img "{target_image_path}" \
                  --output_img "{output_image_path}" \
                  --face_restore \
                  --background_enhance \
                  --face_upsample \
                  --upscale=2 \
                  --codeformer_fidelity=0.5

Starting face swapping from '/content/facesToSwap' to '/content/SwappedFaces' using source face from 'ana.jpg
/content/facesToSwap/frame_190.jpg /content/SwappedFaces/frame_190.jpg
Source image paths: ['/content/ana.jpg']
2026-04-25 03:16:53.161354464 [E:onnxruntime:Default, provider_bridge_ort.cc:2341 TryGetProviderInfo_TensorRT] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1962 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_tensorrt.so with error: libnvinfer.so.10: cannot open shared object file: No such file or directory

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:539 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU require

In [8]:
import os
import sys
from PIL import Image

# Ensure the swapper module can be found
sys.path.insert(0, '/content/inswapper')
from swapper import process

# Define paths
faces_to_swap_dir = "/content/facesToSwap"
swapped_faces_dir = "/content/SwappedFaces"

# Ensure the output directory exists
os.makedirs(swapped_faces_dir, exist_ok=True)

# Define the source image path
source_image_path = "/content/ana.jpg"

# Prepare source_img_paths (a list of paths)
source_img_paths = [source_image_path]

# Define the model path
model_path = "/content/checkpoints/inswapper_128.onnx" # Assuming model_path is defined elsewhere or needs to be here

if os.path.exists(source_image_path):
    print(f"Starting face swapping from '{faces_to_swap_dir}' to '{swapped_faces_dir}' using source face from '{source_image_path}'")

    # Loop through images in the facesToSwap directory
    for filename in os.listdir(faces_to_swap_dir):
        # Check if the file is an image
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            target_image_path = os.path.join(faces_to_swap_dir, filename)
            output_image_path = os.path.join(swapped_faces_dir, filename)

            try:
                print(f"Processing {filename}...")

                # Call the process function with the source image path and target image path
                result_image = process(
                    source_img_paths, # List of source image paths (strings)
                    target_image_path, # Target image path (string)
                    source_indexes=[0], # Assuming first face from source
                    target_indexes=[0], # Assuming first face from target
                    model=model_path
                )
                result_image.save(output_image_path)
                print(f"Successfully swapped face for {filename} and saved to {output_image_path}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    print("Face swapping process completed.")
else:
    print(f"Face swapping process skipped due to missing source image at {source_image_path}.")

Starting face swapping from '/content/facesToSwap' to '/content/SwappedFaces' using source face from '/content/ana.jpg'
Processing frame_190.jpg...
download_path: ./checkpoints/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 95630.61KB/s]


*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:539 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': 

KeyboardInterrupt: 

In [ ]:
import shutil

# Create a zip archive of the SwappedFaces directory
zip_filename = "SwappedFaces_archive"
shutil.make_archive(zip_filename, 'zip', swapped_faces_dir)
print(f"'{swapped_faces_dir}' has been zipped to '{zip_filename}.zip'")